<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain-English rule (three sentences):** A page is worth reviewing if it hasn't been touched in a while, still gets meaningful search visibility, and sits in a position band where a refresh could plausibly move it up rather than being unrecoverable. It skips pages that are already ranking well (no need to review) and pages ranked so deep that a refresh is unlikely to be worth the effort. Everything else gets a reason code explaining exactly why it was skipped.

**Rule conditions:**
- `days_since_last_update >= 180` — stale (roughly 6 months untouched)
- `impressions_90d >= 500` — still visible, an audience exists to benefit from a refresh
- `avg_position != 0` — has real Search Console position data (per `flyrank-data/SKILL.md`, `avg_position == 0` means "no data," not rank zero — treated as missing, never as a real rank)
- `10 < avg_position <= 50` — sits in a recoverable band: not already on page 1 (`<=10`, doesn't need review) and not so deep (`>50`) that refreshing is unlikely to help

**Reason codes:**
- `stale_visible_recoverable_position` — meets all 4 conditions, sent to REVIEW
- `not_stale_enough` — fails the days-since-update condition
- `low_visibility` — fails the impressions condition
- `no_position_data` — `avg_position == 0`
- `already_ranking_well` — `avg_position <= 10`
- `too_deep_to_prioritize` — `avg_position > 50`

In [1]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("days_since_last_update distribution:")
print(df["days_since_last_update"].describe())
print("\nimpressions_90d distribution:")
print(df["impressions_90d"].describe())
print("\navg_position distribution, excluding 0 = no data (1,205 rows per the data dictionary):")
print((df["avg_position"] == 0).sum(), "rows have avg_position == 0")
print(df.loc[df["avg_position"] != 0, "avg_position"].describe())

print("\nRows meeting each individual condition:")
print("  stale (>=180d):", (df["days_since_last_update"] >= 180).sum())
print("  visible (>=500 impr):", (df["impressions_90d"] >= 500).sum())
print("  has position data (!=0):", (df["avg_position"] != 0).sum())
print("  recoverable band (10-50):", ((df["avg_position"] > 10) & (df["avg_position"] <= 50)).sum())


days_since_last_update distribution:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

impressions_90d distribution:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

avg_position distribution, excluding 0 = no data (1,205 rows per the data dictionary):
1205 rows have avg_position == 0
count    28795.000000
mean        17.026268
std         15.152439
min          0.100000
25%          6.700000
50%         11.400000
75%         22.900000
max        245.000000
Name: avg_position, dtype: float64

Rows meeting each individual condition:
  stale (>=180d): 174
  visible (>=500 impr): 16726
  has position data (!=0): 28795
  recoverable band (10-50): 144

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Per `building-baselines/SKILL.md`, the score is a readable multiply/add of simple conditions with **no fitted weights**:

```
score = staleness_factor * visibility_factor * position_factor   (0 if not REVIEW-eligible)
```
where:
- `staleness_factor = min(days_since_last_update / 180, 3.0)` — bigger for staler pages, capped so a 2-year-stale page isn't treated as absurdly more urgent than a 6-month-stale one
- `visibility_factor = log1p(impressions_90d)` — visibility matters with diminishing returns
- `position_factor = clip(1 - (avg_position - 10) / 40, 0, 1)` — favors positions just outside page 1 over near-invisible ones, but only *within* the eligible 10-50 band, never multiplying raw position without limit

After building the queue, this section also evaluates it honestly with **precision@K against the actual label** (`trend_direction == "down"`) and prints the **base rate** next to it, since precision@K means nothing without knowing what a random pick would get — and a **dummy baseline** (majority-class / random) as the floor below the floor, both required by the skill file.

In [2]:


STALE_DAYS = 180
MIN_IMPRESSIONS = 500
POSITION_LOW = 10.0
POSITION_HIGH = 50.0

rule_mask = (
    (df["days_since_last_update"] >= STALE_DAYS) &
    (df["impressions_90d"] >= MIN_IMPRESSIONS) &
    (df["avg_position"] != 0) &
    (df["avg_position"] > POSITION_LOW) &
    (df["avg_position"] <= POSITION_HIGH)
)

df["action"] = np.where(rule_mask, "REVIEW", "SKIP")

def reason_code(row):
    if row["days_since_last_update"] < STALE_DAYS:
        return "not_stale_enough"
    if row["impressions_90d"] < MIN_IMPRESSIONS:
        return "low_visibility"
    if row["avg_position"] == 0:
        return "no_position_data"
    if row["avg_position"] <= POSITION_LOW:
        return "already_ranking_well"
    if row["avg_position"] > POSITION_HIGH:
        return "too_deep_to_prioritize"
    return "stale_visible_recoverable_position"

df["reason_code"] = df.apply(reason_code, axis=1)

staleness_factor = np.minimum(df["days_since_last_update"] / STALE_DAYS, 3.0)
visibility_factor = np.log1p(df["impressions_90d"])
position_factor = np.clip(1 - (df["avg_position"] - POSITION_LOW) / (POSITION_HIGH - POSITION_LOW), 0, 1)

df["score"] = np.where(
    df["action"] == "REVIEW",
    staleness_factor * visibility_factor * position_factor,
    0.0
)

df_ranked = df.sort_values(by=["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)

print("Action counts:")
print(df_ranked["action"].value_counts())
print("\nReason code counts:")
print(df_ranked["reason_code"].value_counts())

# Pitfall check: is the score-order just a disguised sort by impressions or by raw position?
review_sub = df_ranked[df_ranked["action"] == "REVIEW"]
rank_by_score = review_sub.sort_values("score", ascending=False)["content_id"].tolist()
rank_by_impressions = review_sub.sort_values("impressions_90d", ascending=False)["content_id"].tolist()
rank_by_position = review_sub.sort_values("avg_position", ascending=True)["content_id"].tolist()
print("\nIs score-order identical to a plain sort by impressions?", rank_by_score == rank_by_impressions)
print("Is score-order identical to a plain sort by position?", rank_by_score == rank_by_position)

# --- Honest evaluation: precision@K against the actual label, with base rate ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y_true = (df_ranked["trend_direction"] == "down").astype(int).values
base_rate = y_true.mean()

print("\n=== Precision@K evaluation ===")
print(f"Base rate (share of ALL rows that are actually declining): {base_rate:.3f}")
for k in [20, 50, 100]:
    p_at_k = precision_at_k(df_ranked["score"].values, y_true, k)
    print(f"Rule precision@{k}: {p_at_k:.3f}  (vs. base rate {base_rate:.3f}, lift = {p_at_k - base_rate:+.3f})")

# --- Dummy baseline: the floor below the floor ---
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_true), 1)), y_true)
dummy_pred = dummy.predict(np.zeros((len(y_true), 1)))
print(f"\nDummy baseline (predict majority class for everyone) accuracy: {(dummy_pred == y_true).mean():.3f}")
print("The rule's precision@K should clearly beat both the base rate AND this dummy floor to be worth using.")

# Write output CSV. client_id is dropped since it is not needed downstream and the
# self-check requires no client-identifying data in written outputs.
os.makedirs("work/outputs", exist_ok=True)
output_cols = [c for c in df_ranked.columns if c != "client_id"]
df_ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nWrote {len(df_ranked)} rows to work/outputs/baseline_action_score.csv (client_id excluded)")

Action counts:
action
SKIP      29986
REVIEW       14
Name: count, dtype: int64

Reason code counts:
reason_code
not_stale_enough                      29826
low_visibility                          157
stale_visible_recoverable_position       14
already_ranking_well                      3
Name: count, dtype: int64

Is score-order identical to a plain sort by impressions? False
Is score-order identical to a plain sort by position? False

=== Precision@K evaluation ===
Base rate (share of ALL rows that are actually declining): 0.542
Rule precision@20: 0.700  (vs. base rate 0.542, lift = +0.158)
Rule precision@50: 0.480  (vs. base rate 0.542, lift = -0.062)
Rule precision@100: 0.420  (vs. base rate 0.542, lift = -0.122)

Dummy baseline (predict majority class for everyone) accuracy: 0.542
The rule's precision@K should clearly beat both the base rate AND this dummy floor to be worth using.

Wrote 30000 rows to work/outputs/baseline_action_score.csv (client_id excluded)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Every row below is a REVIEW pick under `stale_visible_recoverable_position`, ranked by score. Confidence notes compare CTR against the median CTR for the eligible position band (10-50) — remembering that `ctr` is already a ×100 percentage per `flyrank-data/SKILL.md`, so a value like `0.34` means 0.34%, not 34%. Rows with very low CTR despite real impressions may indicate a search-intent mismatch, not staleness — a refresh may not fix that.

In [3]:

df_out = pd.read_csv("work/outputs/baseline_action_score.csv")

top20 = df_out[df_out["action"] == "REVIEW"].sort_values("score", ascending=False).head(20).copy()

median_ctr_for_band = df_out.loc[
    (df_out["avg_position"] > 10) & (df_out["avg_position"] <= 50), "ctr"
].median()

def confidence_note(row):
    if row["ctr"] < median_ctr_for_band * 0.25:
        return "LOW confidence - CTR far below the typical rate for this position band"
    elif row["ctr"] < median_ctr_for_band:
        return "MEDIUM confidence - CTR somewhat below the band median"
    else:
        return "HIGH confidence - CTR at or above the band median, page gets some clicks"

def what_would_make_it_wrong(row):
    if row["ctr"] == 0:
        return "Zero clicks despite impressions could mean intent mismatch, not staleness"
    if row["content_age_days"] < 60:
        return "Page is relatively young despite being 'stale' by update date"
    return "Assumes decline is due to staleness, not external factors (competitors, SERP changes)"

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(what_would_make_it_wrong, axis=1)

display_cols = ["content_id", "action", "reason_code", "score", "days_since_last_update",
                 "impressions_90d", "avg_position", "ctr", "confidence_note", "what_would_make_it_wrong"]
print(top20[display_cols].to_string(index=False))

print(f"\nMedian CTR for the eligible position band (10-50), for reference: {median_ctr_for_band:.3f}")
print(f"Number of top-20 picks with zero recorded clicks: {(top20['ctr'] == 0).sum()} / 20")

          content_id action                        reason_code     score  days_since_last_update  impressions_90d  avg_position  ctr                                                          confidence_note                                                              what_would_make_it_wrong
content_0a91db491d14 REVIEW stale_visible_recoverable_position 10.054040                     193            13299          10.5 0.49 HIGH confidence - CTR at or above the band median, page gets some clicks Assumes decline is due to staleness, not external factors (competitors, SERP changes)
content_cf56e2e2e282 REVIEW stale_visible_recoverable_position  9.004830                     194            61678          19.7 0.15 HIGH confidence - CTR at or above the band median, page gets some clicks Assumes decline is due to staleness, not external factors (competitors, SERP changes)
content_c2d929d83eaa REVIEW stale_visible_recoverable_position  7.684318                     193             7558          1

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Any row with zero recorded clicks despite hundreds or thousands of impressions is a weak pick — the rule correctly flags it as stale, visible, and in a recoverable position band, but it has no signal for search-intent match. If a large share of the *entire* REVIEW set (not just the top 20) shares this zero-CTR pattern, that's a systemic limitation of the rule, quantified below.

**Leakage check:** The rule and score only use `days_since_last_update`, `impressions_90d`, and `avg_position` — none derived from `trend_direction`, `trend_pct`, or any product-decision flag. No forward-looking windows (`impressions_last_30d`, `impressions_prev_30d`) were used either. This is checked programmatically below, not just asserted in prose.

In [4]:

rule_inputs = {"days_since_last_update", "impressions_90d", "avg_position"}
leakage_risk_cols = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d"
}

overlap = rule_inputs & leakage_risk_cols
print("Rule input columns:", rule_inputs)
print("Known leakage-risk columns:", leakage_risk_cols)
print("Overlap (should be empty):", overlap)
assert len(overlap) == 0, "LEAKAGE DETECTED: the rule uses a label-derived or future-window column!"
print("Leakage check PASSED - the rule only uses columns available at decision time.")

print("\nColumns in the written CSV:")
print(df_out.columns.tolist())
assert "client_id" not in df_out.columns, "client_id should not be present in the output file."
print("Confirmed: client_id is not present in work/outputs/baseline_action_score.csv")

review_set = df_out[df_out["action"] == "REVIEW"]
zero_ctr_share = (review_set["ctr"] == 0).mean()
print(f"\nShare of ALL REVIEW-flagged rows with zero clicks: {zero_ctr_share:.1%}")
print("If this share is large, the zero-CTR weak-pick pattern is systemic, not isolated to the top 20.")

Rule input columns: {'days_since_last_update', 'impressions_90d', 'avg_position'}
Known leakage-risk columns: {'trend_pct', 'trend_direction', 'impressions_last_30d', 'impressions_prev_30d'}
Overlap (should be empty): set()
Leakage check PASSED - the rule only uses columns available at decision time.

Columns in the written CSV:
['content_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.